In [1]:
from neo4j import GraphDatabase

URI = "neo4j+s://eda8932b.databases.neo4j.io"
AUTH = ("eda8932b", "AcIDwM1d-g3xNnqDNyJi8yVer7fx9e8KgVxovc0NiFM")

driver = GraphDatabase.driver(URI, auth=AUTH)

try:
    driver.verify_connectivity()
    print("Successfully connected to Neo4j Aura!")
except Exception as e:
    print(f"Cloud connection failed: {e}")

Successfully connected to Neo4j Aura!


In [ ]:
import pandas as pd
from neo4j import GraphDatabase

def load_graph_data():
    print("Loading CSVs into memory...")
    nodes_df = pd.read_csv("nodes.csv")
    edges_df = pd.read_csv("edges.csv")

    nodes_df = nodes_df.fillna("")
    edges_df = edges_df.fillna("")

    driver = GraphDatabase.driver(URI, auth=AUTH)

    with driver.session() as session:
        
        # --- LOADING THE NODES ---
        print("Starting Node Insertion...")
        
        for label, group in nodes_df.groupby('label'):
            batch = group.to_dict('records')

            node_query = f"""
            UNWIND $batch AS row
            MERGE (n:Entity:{label} {{id: row.id}})
            SET n.name = row.name
            """
            session.run(node_query, batch=batch)
            print(f"Loaded {len(batch)} nodes of type: {label}")

        # CREATING INDEXES
        print("Creating index on Entity id...")
        session.run("CREATE INDEX entity_id IF NOT EXISTS FOR (n:Entity) ON (n.id)")

        # --- LOADING EDGES ----
        print("Starting Edge Insertion...")
        for relation, group in edges_df.groupby('relation'):
            batch = group.to_dict('records')
            
            edge_query = f"""
            UNWIND $batch AS row
            MATCH (s:Entity {{id: row.source}})
            MATCH (t:Entity {{id: row.target}})
            MERGE (s)-[r:{relation}]->(t)
            """
            session.run(edge_query, batch=batch)
            print(f"Loaded {len(batch)} relationships of type: {relation}")

    driver.close()
    print("Graph Data Successfully Loaded into Aura!")

load_graph_data()

Loading CSVs into memory...
Starting Node Insertion...
Loaded 1210 nodes of type: Chemical
Loaded 1564 nodes of type: Disease
Loaded 5228 nodes of type: TextChunk
Creating index on Entity id...
Starting Edge Insertion...
Loaded 3053 relationships of type: CO_OCCURS_WITH
Loaded 8732 relationships of type: MENTIONED_IN
Graph Data Successfully Loaded into Aura!


: 